# InternVL3-2B Evaluation on PHLOP Dataset

In [ ]:
%pip install transformers accelerate
%pip install decord opencv-python matplotlib
%pip install datasets huggingface_hub
%pip install flash-attn --no-build-isolation

## Configuration

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import login
from decord import VideoReader, cpu

sys.path.insert(0, str(Path(".").resolve()))

from phlop_eval_common import (
    load_phlop_splits, load_json_file, load_qa_from_path,
    get_physical_props, get_taxonomy,
    evaluate_response_quality, build_dynamic_prompt,
    save_and_score_results, EVAL_OPTIONS,
    FINE_TUNE_CONFIGS, get_val_difficulty_filter, filter_qa_by_difficulty,
    build_training_index,
)

REPO_ID = "zimmari-ai/phlop"
HF_TOKEN = os.environ.get("HF_TOKEN", True)
CAMERA_MODE = "static"
NUM_FRAMES = 32
MAX_SAMPLES = 4000

In [ ]:
if isinstance(HF_TOKEN, str) and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)

## Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_path = "OpenGVLab/InternVL3-2B"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2",
).eval()

## Video Loading Helpers

In [ ]:
def get_index(num_frames, num_segments):
    seg_size = float(num_frames - 1) / num_segments
    start = int(seg_size / 2)
    offsets = np.array([start + int(np.round(seg_size * idx)) for idx in range(num_segments)])
    return offsets


def load_video(video_path, num_segments=8):
    vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
    num_frames = len(vr)
    frame_indices = get_index(num_frames, num_segments)
    frames = vr.get_batch(frame_indices).asnumpy()
    return frames


def preprocess_video_frames(frames, resolution=448):
    frames = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    frames = (frames - mean) / std
    frames = torch.nn.functional.interpolate(
        frames, size=(resolution, resolution), mode="bicubic", align_corners=False
    )
    return frames

## Load Dataset from HuggingFace

In [ ]:
splits = load_phlop_splits(REPO_ID, token=HF_TOKEN)
test_ds = splits["test"]
for name, ds in splits.items():
    print(f"  {name}: {len(ds)} scenes")

## Evaluation

In [ ]:
results = []

for idx in tqdm(range(min(MAX_SAMPLES, len(test_ds))), desc="Evaluating"):
    sample = test_ds[idx]
    
    video_path = (sample.get("videos") or {}).get(CAMERA_MODE)
    meta_path = (sample.get("metadata") or {}).get(CAMERA_MODE)
    if not video_path or not meta_path:
        continue

    metadata = load_json_file(meta_path)
    physical_props = get_physical_props(metadata)
    taxonomy = get_taxonomy(metadata)
    qa_list = load_qa_from_path((sample.get("qa") or {}).get(CAMERA_MODE))
    if not qa_list:
        continue

    scene_id = sample.get("id", str(idx))

    video_tensor = load_video(video_path, num_segments=NUM_FRAMES)
    video_tensor = preprocess_video_frames(video_tensor, resolution=448)
    video_tensor = video_tensor.to(model.device).half()

    for option in EVAL_OPTIONS:
        tax = taxonomy if option["is_taxonomy"] else {}
        phys = physical_props if option["is_physics"] else {}

        for qa in qa_list:
            prompt = build_dynamic_prompt(
                taxonomy=tax,
                physical_props=phys,
                question=qa["question"],
                options=qa.get("options"),
                explanation=qa.get("explanation"),
                num_frames=NUM_FRAMES,
            )

            try:
                with torch.no_grad():
                    response = model.chat(
                        tokenizer,
                        video_tensor,
                        question=prompt,
                        generation_config={
                            "do_sample": True,
                            "max_new_tokens": 200,
                            "temperature": 0.1,
                            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
                        },
                        history=None,
                    )
            except RuntimeError as e:
                print(f"Error at idx {idx}: {e}")
                response = "Error"

            eval_res = evaluate_response_quality(response, qa["answer"], taxonomy, physical_props)

            results.append({
                "scene_id": scene_id,
                "question": qa["question"],
                "answer": qa["answer"],
                "options": qa.get("options"),
                "response": response,
                "correct": eval_res["correct"],
                "error": eval_res["error"],
                "option": option["name"],
            })

print(f"Collected {len(results)} results")

## Results

In [ ]:
os.makedirs("results", exist_ok=True)
scores = save_and_score_results(results, "results/internvl3_2b_results.json")

## Part 2: Fine-tuning

LoRA fine-tuning with 4 difficulty configurations:
1. **easy** - train on easy, validate on rest
2. **easy_medium** - train on easy+medium, validate on rest
3. **hard** - train on hard, validate on rest
4. **full** - full dataset

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset

OUTPUT_DIR = "./internvl3_checkpoints"
MAX_STEPS = 50


class PHLOPFineTuneDataset(TorchDataset):
    """Expands scenes into (scene, question) pairs with difficulty filtering."""

    def __init__(self, ds, index, camera_mode="static", num_segments=32):
        self.ds = ds
        self.index = index
        self.camera_mode = camera_mode
        self.num_segments = num_segments

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        scene_idx, qa_idx = self.index[idx]
        sample = self.ds[scene_idx]

        video_path = (sample.get("videos") or {}).get(self.camera_mode)
        meta_path = (sample.get("metadata") or {}).get(self.camera_mode)
        metadata = load_json_file(meta_path) if meta_path else {}
        physical_props = get_physical_props(metadata)
        taxonomy = get_taxonomy(metadata)

        qa_list = load_qa_from_path((sample.get("qa") or {}).get(self.camera_mode))
        qa = qa_list[qa_idx] if qa_idx < len(qa_list) else {}

        question = qa.get("question", "")
        answer = qa.get("answer", "")
        if isinstance(answer, list):
            answer = ", ".join(str(a) for a in answer)

        prompt = build_dynamic_prompt(
            taxonomy=taxonomy, physical_props=physical_props,
            question=question, options=qa.get("options"),
            explanation=qa.get("explanation"), num_frames=self.num_segments,
        )

        video_tensor = None
        if video_path:
            frames = load_video(video_path, num_segments=self.num_segments)
            video_tensor = preprocess_video_frames(frames, resolution=448)

        return {
            "video_tensor": video_tensor,
            "prompt": prompt,
            "answer": str(answer),
        }


class InternVL3DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        prompts = [item["prompt"] + "\n" + item["answer"] for item in batch]
        encodings = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        encodings["labels"] = encodings["input_ids"].clone()

        videos = [item["video_tensor"] for item in batch if item["video_tensor"] is not None]
        if videos:
            encodings["pixel_values"] = torch.cat(videos, dim=0)
        return encodings

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none", task_type="CAUSAL_LM",
)

saved_checkpoints = []
train_ds = splits.get("train", splits["test"])

for cfg_name, cfg in FINE_TUNE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Fine-tuning config: {cfg_name}")
    val_diff = get_val_difficulty_filter(cfg["train_difficulty"]) if cfg["val_on_rest"] else None
    print(f"  Train: {cfg['train_difficulty']}, Val: {val_diff}")
    print(f"{'='*60}")

    train_index = build_training_index(train_ds, difficulty_filter=cfg["train_difficulty"], camera_mode=CAMERA_MODE)
    if not train_index:
        print(f"  Skipping {cfg_name}: no training samples.")
        continue

    ft_train = PHLOPFineTuneDataset(train_ds, train_index, camera_mode=CAMERA_MODE, num_segments=NUM_FRAMES)

    ft_model = AutoModel.from_pretrained(
        model_path, trust_remote_code=True,
        torch_dtype=torch.float16, device_map="auto",
        attn_implementation="flash_attention_2",
    )
    ft_model = get_peft_model(ft_model, lora_config)
    ft_model.print_trainable_parameters()
    ft_model.train()

    ckpt_dir = os.path.join(OUTPUT_DIR, cfg_name)
    training_args = TrainingArguments(
        output_dir=ckpt_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        max_steps=MAX_STEPS,
        fp16=True,
        logging_steps=10,
        save_steps=500,
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=ft_model, args=training_args,
        train_dataset=ft_train,
        data_collator=InternVL3DataCollator(tokenizer),
    )
    trainer.train()
    trainer.save_model()
    saved_checkpoints.append((cfg_name, ckpt_dir))
    del ft_model
    torch.cuda.empty_cache()

print(f"\nSaved {len(saved_checkpoints)} checkpoints: {saved_checkpoints}")

## Part 3: Evaluate Fine-tuned Models on Test

In [ ]:
from peft import PeftModel

print(f"\n{'Config':<20} {'Accuracy':>10}")
print("-" * 35)

for cfg_name, ckpt_dir in saved_checkpoints:
    ft_model = AutoModel.from_pretrained(
        model_path, trust_remote_code=True,
        torch_dtype=torch.float16, device_map="auto",
        attn_implementation="flash_attention_2",
    )
    ft_model = PeftModel.from_pretrained(ft_model, ckpt_dir)
    ft_model.eval()

    ft_results = []
    for idx in tqdm(range(min(MAX_SAMPLES, len(test_ds))), desc=f"Eval {cfg_name}"):
        sample = test_ds[idx]
        video_path = (sample.get("videos") or {}).get(CAMERA_MODE)
        meta_path = (sample.get("metadata") or {}).get(CAMERA_MODE)
        if not video_path or not meta_path:
            continue

        metadata = load_json_file(meta_path)
        physical_props = get_physical_props(metadata)
        taxonomy = get_taxonomy(metadata)
        qa_list = load_qa_from_path((sample.get("qa") or {}).get(CAMERA_MODE))
        if not qa_list:
            continue

        video_tensor = load_video(video_path, num_segments=NUM_FRAMES)
        video_tensor = preprocess_video_frames(video_tensor, resolution=448)
        video_tensor = video_tensor.to(ft_model.device).half()

        for qa in qa_list:
            prompt = build_dynamic_prompt(
                taxonomy=taxonomy, physical_props=physical_props,
                question=qa["question"], options=qa.get("options"),
                explanation=qa.get("explanation"), num_frames=NUM_FRAMES,
            )
            try:
                with torch.no_grad():
                    response = ft_model.chat(
                        tokenizer, video_tensor, question=prompt,
                        generation_config={"do_sample": True, "max_new_tokens": 200, "temperature": 0.1,
                                           "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id},
                        history=None,
                    )
            except RuntimeError as e:
                response = "Error"

            eval_res = evaluate_response_quality(response, qa["answer"], taxonomy, physical_props)
            ft_results.append({"correct": eval_res["correct"], "option": "taxonomy_and_physics"})

    if ft_results:
        acc = sum(r["correct"] for r in ft_results) / len(ft_results)
        print(f"{cfg_name:<20} {acc:>10.4f} ({sum(r['correct'] for r in ft_results)}/{len(ft_results)})")

    os.makedirs("results", exist_ok=True)
    save_and_score_results(ft_results, f"results/internvl3_ft_{cfg_name}.json")

    del ft_model
    torch.cuda.empty_cache()